In [ ]:
import numpy as np
import pandas as pd
import os
from datetime import datetime, date
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.vector_ar.vecm import VECM, coint_johansen
import scipy.linalg as la
from scipy.linalg import eig
from statsmodels.tsa.vector_ar.var_model import VAR
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
print(f"modules have been imported.")

In [ ]:
# Step 1: Lets grab the historical time series from Mohaddes first. 
# Question should we grab it by economies first? if yes then lets take the 'Country Data'
print(os.getcwd())
target_filename = "Country Data (1979Q2-2023Q3).xls"
target_file_path = os.getcwd() + "\\" + "GVAR Database (1979Q2-2023Q3)\\" + target_filename
print(f"target_file_path is: {target_file_path}")
# dataframe the country data
countries_df = pd.read_excel(target_file_path, sheet_name = None)
countries_df.keys()


In [ ]:
us_df = countries_df['USA']
sg_df = countries_df['SINGAPORE']
cn_df = countries_df['CHINA']
eu_df = countries_df['EURO']
# convert the date into datetime format 
# ie: 1979Q2 is 30-Jun-1979.
us_df['date_2'] = pd.PeriodIndex(us_df['date'], freq = 'Q').to_timestamp(how = "end").date
sg_df['date_2'] = pd.PeriodIndex(sg_df['date'], freq = 'Q').to_timestamp(how = "end").date
cn_df['date_2'] = pd.PeriodIndex(cn_df['date'], freq = 'Q').to_timestamp(how = "end").date
eu_df['date_2'] = pd.PeriodIndex(eu_df['date'], freq = 'Q').to_timestamp(how = "end").date


us_target_cols = [col for col in us_df.columns if 'date' not in col]
us_target_cols = ['date_2'] + us_target_cols 

sg_target_cols = [col for col in sg_df.columns if 'date' not in col]
sg_target_cols = ['date_2'] + sg_target_cols 

cn_target_cols = [col for col in cn_df.columns if 'date' not in col]
cn_target_cols = ['date_2'] + cn_target_cols 

eu_target_cols = [col for col in eu_df.columns if 'date' not in col]
eu_target_cols = ['date_2'] + eu_target_cols 

print(us_target_cols)

us_df = us_df[us_target_cols]
us_df = us_df.rename(columns={'date_2': 'date'})
us_df

sg_df = sg_df[sg_target_cols]
sg_df = sg_df.rename(columns={'date_2': 'date'})
sg_df

cn_df = cn_df[cn_target_cols]
cn_df = cn_df.rename(columns={'date_2': 'date'})
cn_df

eu_df = eu_df[eu_target_cols]
eu_df = eu_df.rename(columns={'date_2': 'date'})
eu_df

In [ ]:
# 1. We first and foremostly must do is to do the ADF (augmented dickey fuller) test to check on its unit root testing. 
## 2. We need to do the AIC and BIC test to find out the lags that are most optimal for the VARX
## 3. This is to test the weak exogeneity for each variable components in the us_df table. 

def run_gvar_stationary_check(country, dataframe, significance_level = 0.05):
    adf_results = []
    # 2. Automated ADF testing loop function
    for column in dataframe.columns:
        if column not in 'date':
            
            # --- TEST 1: Check Raw Levels ---
            # autolag = 'AIC' lets the module find the best lag length to remove serial autocorrelation.
            # regression = 'c' includes a constant intercept in the test equation.
            # let significance_level = 0.05
            level_res = adfuller(dataframe[column], autolag = 'AIC', regression = 'c')
            level_pvalue = level_res[1]
            level_stationary = level_pvalue < significance_level
            
#            print(f"--- Results for the US {column} ---")
#            print(f"level_res is: {level_res}")
#            print(f"level_pvalue is: {level_pvalue}")
#            print(f"level_stationary is: {level_stationary}")
            
            # Determine Integration Order Classification
            if not level_stationary:
                classification = "I(1) - Perfect for VECM"
            elif level_stationary:
                classification = "I(0) - Already Stationary"
            else:
                classification = "I(2) or Higher/ Explosive"
            
            adf_results.append({
                'country' : country,
                'variable' : column,
                'level_pvalue' : round(level_pvalue, 4),
                'level_stationary' : level_stationary,
                'classification' : classification,
            })
        
    return pd.DataFrame(adf_results)

us_adfuller_df = run_gvar_stationary_check("US", us_df, significance_level = 0.05)
sg_adfuller_df = run_gvar_stationary_check("SG", sg_df, significance_level = 0.05)
cn_adfuller_df = run_gvar_stationary_check("CN", cn_df, significance_level = 0.05)
eu_adfuller_df = run_gvar_stationary_check("EU", eu_df, significance_level = 0.05)

all_adfuller_df = us_adfuller_df.copy()
all_adfuller_df = pd.concat([all_adfuller_df, sg_adfuller_df, cn_adfuller_df, eu_adfuller_df], axis = 0)

display(all_adfuller_df)

# The code below is supposed to be a module to find the best lag order based on AIC, BIC and HQIC. 
# However, seeing that we need to see the results of the VECMX. I will get back to this when i can have a better picture on how the lags p, q = 2,2 works. 

In [ ]:
# this code block will be used for the AIC, BIC and HQIC

endog_vars = ['y', 'Dp', 'eq', 'r', 'lr', 'poil', 'pmat', 'pmetal']
exog_vars = ['ys', 'Dps']

us_endog_df = us_df[endog_vars]
us_exog_df = us_df[exog_vars]

display(us_endog_df)
display(us_exog_df)

# define variables
exog_df = us_exog_df
endog_df = us_endog_df
#display(exog_df)
#print(f"endog_df is: \n {display(endog_df)}")

# Grid parameters
max_p = 3 # for domestic/ endog vars
max_q = 2 # for foreign/ exog vars
max_lag = max(max_p, max_q)

results = []

# 2. High level optimization loop using statsmodels internals. 
for p in range(1, max_p + 1):
    for q in range(0, max_q + 1):
        
        print(f"p,q are: ({p},{q})")
        # Construct foreign variables lag columns
        exog_list = []
        for lag in range(0, q + 1):
            print(f"lag (q) now is: {lag}")
            target_exog_df = exog_df.shift(lag)
            if lag >= 1:
                target_exog_df.columns = [f"{col}_lag{q}" for col in target_exog_df.columns]

            exog_list.append(target_exog_df) # shift the lag based on q variable.      
        
        X_exog = pd.concat(exog_list, axis = 1).dropna() # to concatenate by columns. 
#        print(f"X_exog is:\n {display(X_exog)}")
#        print(f"X_exog.index is:\n {display(X_exog.index)}")

        # Align endog target dataframe to match exogenous shift row losses.
        Y_target = endog_df.loc[X_exog.index]
#        print(f"Y_target is:\n {display(Y_target)}")

        # CRUCIAL GVAR TRIMMING: Ensure ALL lag combinations run on the exact same time window. 
        # This keeps the comparison mathematically fair across the loops. 
        X_exog_clean = X_exog.iloc[(max_lag - p):] # max_lag = 3, p = 1, ans = 2 
        Y_target_clean = Y_target.iloc[(max_lag - p):] # max_lag = 3, p = 1, ans = 2

#        print(f"X_exog_clean is:\n {display(X_exog_clean)}")
        print(f"X_exog_clean shape is:\n {(X_exog_clean.shape)}")
#        print(f"Y_target_clean is:\n {display(Y_target_clean)}")
        print(f"Y_target_clean shape is:\n {(Y_target_clean.shape)}")

        # 3. Call the built-in statsmodels VAR function with exog parameters
        # maxlags = p forces the engine to fit exactly 'p' for domestic lags
        model = VAR(endog = Y_target_clean, exog = X_exog_clean)
        results_fitted = model.fit(maxlags=p)

        # 4. Pull the native optimized AIC/BIC/HQIC calculations from the results object.
        results.append({
            'p' : p,
            'q' : q,
            'AIC' : results_fitted.aic,
            'BIC' : results_fitted.bic,
            'HQIC' : results_fitted.hqic,
        })
        
#        if q == 1: 
#            print(f"lets look at the results first.")
#            break

# 5. Display the final score sheet.
df_scores = pd.DataFrame(results)
print("--- Statsmodels Evaluated Scores ---")
print(f"df_scores is: \n {display(df_scores)}")
print(df_scores.to_string(index=False))

# Extract the winning combinations
print(f"\nWinning Architecture by AIC: P={df_scores.loc[df_scores['AIC'].idxmin(), 'p']}, Q = {df_scores.loc[df_scores['AIC'].idxmin(), 'q']}")
print(f"\nWinning Architecture by BIC: P={df_scores.loc[df_scores['BIC'].idxmin(), 'p']}, Q = {df_scores.loc[df_scores['BIC'].idxmin(), 'q']}")
print(f"\nWinning Architecture by HQIC: P={df_scores.loc[df_scores['HQIC'].idxmin(), 'p']}, Q = {df_scores.loc[df_scores['HQIC'].idxmin(), 'q']}")

$ \Large\text{We are working on the VECMX* (1,1) based from the VARX(2,2)} $

$ \Large\text{Here is the formula below so that we can understand which formulas we are going to keep as levels (contemporary and lagged) and change in variables (contemporary and lagged.)} $

$ \Large\text{VARX* is:  } x_{i,t} = a_{i,0} + \Phi_{i,1} x_{i, t-1} + \Phi_{i,2} x_{i,t-2} + \Lambda_{i,0} x^*_{i,t} + \Lambda_{i,1} x^*_{i, t-1} + \Lambda_{i,2} x^*_{i,t-2} + \epsilon_{i,t} $

$ \Large\text{VECMX* is:  } \Delta x_{i,t} = a_{i,0} + \Gamma_{i,1} \Delta x_{i, t-1} + \Psi_{i,0} \Delta x^*_{i,t} + \Psi_{i,1} \Delta x_{i,t-1} + \Pi_{i} z_{i,t-1} + \epsilon_{i,t} $

$ \Large\text{where: } $

$ \Large\text{1. } \Delta x_{i,t} = x_{i,t} - x_{i,t-1} $

$ \Large\text{2. } \Delta x_{i,t-1} = x_{i,t-1} - x_{i,t-2} $

$ \Large\text{3. } \Delta z_{i,t-1} = \begin{bmatrix} x_{i,t-1} \\ x^*_{i,t-1} \end{bmatrix} $

In [ ]:
# ok so for the US model below, lets follow the simple example of VARX(2,2) and VECMX(1,1)
# p, q = 2, 2

# Defining the variables for domestic vs. foreign.
endog_vars = ['y', 'Dp', 'eq', 'r', 'lr', 'poil', 'pmat', 'pmetal']
exog_vars = ['ys', 'Dps']

us_endog_df = us_df[endog_vars]
us_exog_df = us_df[exog_vars]

display(us_endog_df)
#display(us_exog_df)

# lets assume that our lags are as follows p_{domestic} = 2, q_{foreign} = 2
# so we are now finding for VAR(p, q) = VAR(2, 2)
# -- USER DEFINED PARAMETERS -- # 
# Once when we are done with the testing we can remove the user defined parameters and use the AIC test to indicate the lags
p, q = 2, 2

max_lag = max(p,q)

# to define the exog data here based on lags (defined by AIC/BIC/HQIC) and shifts (based on also lags?).
us_exog_contemp_df = us_exog_df.iloc[max_lag:] # align to match the loss of p domestic lags.
us_exog_lag1_df = us_exog_df.shift(1).iloc[max_lag:].add_suffix('_lag1') # lag levels of 1
us_exog_lag2_df = us_exog_df.shift(2).iloc[max_lag:].add_suffix('_lag2')  # lag levels of 2
print(f"us_exog_contemp_df is: \n {display(us_exog_contemp_df)}")
print(f"us_exog_lag1_df is: \n {display(us_exog_lag1_df)}")
print(f"us_exog_lag2_df is: \n {display(us_exog_lag2_df)}")

#print(f"us_exog_df shift(1) is: {display(us_exog_df.shift(1))}")
#print(f"us_exog_df shift(1).iloc[p:] is: {display(us_exog_df.shift(1).iloc[p:])}")
#print(f"us_exog_df shift(2) is: {display(us_exog_df.shift(2))}")
#print(f"us_exog_df shift(2).iloc[p:] is: {display(us_exog_df.shift(2).iloc[p:])}")

# Concatenate the exog data here with the shifts and lags. 
us_exog_final_df = pd.concat([us_exog_contemp_df, us_exog_lag1_df, us_exog_lag2_df], axis = 1)
print(f"us_exog_final_df is: {display(us_exog_final_df)}")

# Add in the constant from the exog data. 
us_exog_final_df = sm.add_constant(us_exog_final_df)
print(f"us_exog_final_df after including constant is: {display(us_exog_final_df)}")

# define us endogeneous data
us_endog_final_df = us_endog_df.iloc[p:]
print(f"us_endog_final_df is: \n {display(us_endog_final_df)}")

#--- VECMX Data Prep ---#
# This is where we switch it from VARX(2,2) = VECMX(1,1)
# lets compute the \delta x_{i,t} = x_{i,t} - x_{i,t-1} --> dependent variable 
us_endog_vecm_comtemp_df = us_df[endog_vars]
us_endog_vecm_lag1_df = us_df[endog_vars].shift(1).iloc[max_lag:].add_suffix('_lag1')
us_endog_vecm_lag2_df = us_df[endog_vars].shift(2).iloc[max_lag:].add_suffix('_lag2')
us_endog_vecm_diff1_df = us_df[endog_vars].diff().add_suffix('_diff1')
us_endog_vecm_diff2_df = us_df[endog_vars].shift(1).diff().add_suffix('_diff2')

print(f"us_endog_vecm_comtemp_df is: \n {display(us_endog_vecm_comtemp_df)}")
print(f"us_endog_vecm_lag1_df is: \n {display(us_endog_vecm_lag1_df)}")
print(f"us_endog_vecm_lag2_df is: \n {display(us_endog_vecm_lag2_df)}")
print(f"us_endog_vecm_diff1_df is: \n {display(us_endog_vecm_diff1_df)}")
print(f"us_endog_vecm_diff2_df is: \n {display(us_endog_vecm_diff2_df)}")

us_endog_vecm_final_df = pd.concat([us_endog_vecm_comtemp_df, us_endog_vecm_lag1_df, us_endog_vecm_lag2_df, us_endog_vecm_diff1_df, us_endog_vecm_diff2_df], axis = 1)
print(f"us_endog_vecm_df is: \n {display(us_endog_vecm_final_df)}")

us_exog_final_df = pd.concat([us_exog_contemp_df, us_exog_lag1_df, us_exog_lag2_df], axis = 1)

us_final_vecm_df = pd.concat([us_endog_vecm_final_df, us_exog_final_df], axis = 1).dropna()
print(f"us_final_vecm_df is: \n {display(us_final_vecm_df)}")

# sort the columns
us_final_vecm_df = us_final_vecm_df.sort_index(axis = 1)
print(f"us_final_vecm_df after sorting is: \n {display(us_final_vecm_df)}")

# ---- TO NOTE: WE DO NOT NEED TO RUN VARX FIRST, WE CAN GO AHEAD AND DO VECMX FIRST. ----- #
# Fit the level VARX model via statsmodels 
# we used maxlags = max(p,q) for the domestic variables and passing foreign vectors as exog.
varx_model = VAR(endog = us_endog_final_df, exog = us_exog_final_df)
varx_results = varx_model.fit(maxlags = p)

print(f"--- US VARX({p},{q}) Model Estimated Successfully ---")
print(f"Endogenous Matrix Dimensions (Variables): {us_endog_final_df.shape[1]}")
print(f"Exogenous Matrix Dimensions (Regressors): {us_exog_final_df.shape[1]}")
print(f"\nCoefficient Parameters shape for the US system:", varx_results.params.shape)
print(varx_results.summary())

In [ ]:
# Use Johansens test to find the cointegration (Beta) and speed of adjustment (alpha)
# lets remove the global variables from the us_df data so that we can only find the cointegration 
# relationship between the domestic vars and its foreign vars.

print(us_df.columns)

endog_vars = ['y', 'Dp', 'eq', 'r', 'lr']
exog_vars = ['ys', 'Dps', 'eps']

us_raw_level_df = pd.concat([us_df[endog_vars], us_df[exog_vars]], axis = 1)
display(us_raw_level_df)
print(f"columns for us_raw_level_df is: {us_raw_level_df.columns}")

# Run the automated Johansens test
# det_order = 0, includes an unrestricted constant intercept in the VECM space. 
# k_ar_diff = 1, enforces 1 lag of short run differences (corresponds to p = 2 for levels).
johansen_res = coint_johansen(us_raw_level_df, det_order = 0, k_ar_diff = 0)

# Rearrange the data to compare the johansens trace stats vs the crit values properly.
ci_level_df = pd.DataFrame(johansen_res.cvt, index = [i for i in range(len(johansen_res.cvt))], columns = ['ci_90%', 'ci_95%', 'ci_99%'])
trace_stats_df = pd.DataFrame(johansen_res.lr1, columns = ['trace_stats'])
max_eigenvalues_df = pd.DataFrame(johansen_res.lr2, columns = ['max_eigenvalue'])
johansen_df = pd.concat([max_eigenvalues_df, trace_stats_df, ci_level_df], axis = 1)
johansen_df['id_90%'] = np.where(johansen_df['trace_stats'] >= johansen_df['ci_90%'],1,0)
johansen_df['id_95%'] = np.where(johansen_df['trace_stats'] >= johansen_df['ci_95%'],1,0)
johansen_df['id_99%'] = np.where(johansen_df['trace_stats'] >= johansen_df['ci_99%'],1,0)
johansen_df['rank_90%'] = johansen_df['id_90%'].cumsum()
johansen_df['rank_95%'] = johansen_df['id_95%'].cumsum()
johansen_df['rank_99%'] = johansen_df['id_99%'].cumsum()


# Extracts the output from the results of the Johansen test. 
print("--- Automation Johansen Statistics ---")
print("Trace Statistics: \n", johansen_res.lr1)
print("\nMax Eigenvalue Statistics:\n", johansen_res.lr2)
print("Critical values (90%, 95%, 99% thresholds):\n", johansen_res.cvt)
print(f"final table for the johansen statistics to see the ranking is as below:")
print(display(johansen_df))